In [ ]:
!pip install earthengine-api
!earthengine authenticate

E0000 00:00:1748071882.627848     702 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1748071882.639128     702 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
Authenticate: Limited support in Colab. Use ee.Authenticate() or --auth_mode=notebook instead.
W0524 07:31:27.754780 140126914756608 _default.py:711] No project ID could be determined. Consider running `gcloud config set project` or setting the GOOGLE_CLOUD_PROJECT environment variable
To authorize access needed by Earth Engine, open the following URL in a web browser and follow the instructions. If the web browser does not start automatically, please manually browse the URL below.

    https://code.earthengine.google.com/client-auth?scopes=https%3A//www.googleapis.com/auth/earthengine%20https%3A//www.googleapis.com/auth/cloud-platform%20h

In [ ]:
### INITIALIZE VARIABLES ###
project_name = "ee-ndatest" # Name of the project on your google cloud project
drive_folder= "Province_New" # Name of the folder in your google drive where the data is stored
numRectsX = 4  # Number of rectangles along the X axis
numRectsY = 3 # Number of rectangles along the Y axis
# .tif

In [ ]:
import ee

# Authenticate and initialize the Earth Engine API
ee.Authenticate()
ee.Initialize(project=project_name)

# Hoang Hoa polygon (bounding box)
hoanghoa_poly = [[105.90930938720709, 20.035177230835075],
 [105.94056701660156, 20.035177230835075],
  [105.94056701660156, 20.00537490844738],
   [105.90930938720709, 20.00537490844738],
    [105.90930938720709, 20.035177230835075]]

hoanghoa_polygon = ee.Geometry.Polygon([hoanghoa_poly])

# Cloud mask for Sentinel-2
def maskS2clouds(image):
    qa = image.select('QA60')
    cloudBitMask = 1 << 10
    cirrusBitMask = 1 << 11
    mask = qa.bitwiseAnd(cloudBitMask).eq(0).And(qa.bitwiseAnd(cirrusBitMask).eq(0))
    return image.updateMask(mask).divide(10000)

# Export function
def export_image(place_name: str, region, from_date: str, to_date: str):
    image = ee.ImageCollection('COPERNICUS/S2_HARMONIZED') \
        .filterBounds(region) \
        .filterDate(from_date, to_date) \
        .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20)) \
        .map(maskS2clouds) \
        .median() \
        .select(['B1', 'B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A', 'B9', 'B10', 'B11', 'B12']) \
        .clip(region)

    description = f"{place_name}_FULL"

    task = ee.batch.Export.image.toDrive(
        image=image,
        description=description,
        folder=drive_folder,
        scale=10,  # Recommended for Sentinel-2
        region=region,
        maxPixels=1e13,
        fileFormat='GeoTIFF',
        formatOptions={'cloudOptimized': True}
    )

    task.start()
    print(f"Exporting: {description}")

# Example usage (adjust month/year as needed)
for month in range(5, 6):  # Only May for this example
    if month == 12:
        from_date = f'2023-{month:02d}-01'
        to_date = '2023-12-31'
    else:
        from_date = f'2023-{month:02d}-01'
        to_date = f'2023-{month + 1:02d}-01'

    export_image("HoangHoa", hoanghoa_polygon, from_date, to_date)

MessageError: Error: credential propagation was unsuccessful

## Fixed

In [ ]:
import ee
import os
import ast
import unicodedata
import re

# === KHỞI TẠO EARTH ENGINE ===
ee.Authenticate()
ee.Initialize(project=project_name)

# === CẤU HÌNH ===
count = 0
polygon_folder = '/content/polygon'  # 🔁 Cập nhật thư mục của bạn
from_date = '2024-01-01'
to_date = '2024-12-31'
suffix_to_strip = ''

# === HÀM XỬ LÝ TÊN ===
def strip_accents(text):
    text = text.replace("Đ", "D").replace("đ", "d")
    return ''.join(
        c for c in unicodedata.normalize('NFKD', text)
        if not unicodedata.combining(c)
    )

def sanitize_description(name):
    name = strip_accents(name)
    return re.sub(r'[^a-zA-Z0-9.,:_-]', '_', name)[:100]

# === HÀM MASK MÂY ===
def maskS2clouds(image):
    qa = image.select('QA60')
    cloudBitMask = 1 << 10
    cirrusBitMask = 1 << 11
    mask = qa.bitwiseAnd(cloudBitMask).eq(0).And(qa.bitwiseAnd(cirrusBitMask).eq(0))
    return image.updateMask(mask).divide(10000)

# === HÀM EXPORT ===
def export_image(place_name: str, region, from_date: str, to_date: str):
    global count
    description = sanitize_description(f"{place_name}_full_2024")

    image = ee.ImageCollection('COPERNICUS/S2_HARMONIZED') \
        .filterBounds(region) \
        .filterDate(from_date, to_date) \
        .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 100)) \
        .map(maskS2clouds) \
        .median() \
        .select(['B1', 'B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A', 'B9', 'B10', 'B11', 'B12']) \
        .clip(region)

    task = ee.batch.Export.image.toDrive(
        image=image,
        description=description,
        folder=drive_folder,
        scale=10,
        region=region,
        maxPixels=1e13,
        fileFormat='GeoTIFF',
        formatOptions={'cloudOptimized': True}
    )

    task.start()
    print(f"🚀 Bắt đầu xuất: {description}")
    count += 1

# === DUYỆT QUA CÁC FILE POLYGON ===
for filename in os.listdir(polygon_folder):
    if filename.endswith('.txt'):
        base_name = os.path.splitext(filename)[0]

        # Tên không dấu cho xuất file
        place_name = base_name.replace(suffix_to_strip, '') if base_name.endswith(suffix_to_strip) else base_name
        clean_place_name = strip_accents(place_name)

        # Đọc polygon
        txt_path = os.path.join(polygon_folder, filename)
        with open(txt_path, 'r', encoding='utf-8') as f:
            coords = ast.literal_eval(f.read())

        # Tạo geometry và export
        region = ee.Geometry.Polygon([coords])
        export_image(clean_place_name, region, from_date, to_date)

print(count)

🚀 Bắt đầu xuất: HaNoi_full_2024
1


## Air 360 task

In [ ]:
import ee
import json

project_name = "linear-sight-456113-t3"  # Your GEE project
drive_folder = "SateImg1km_2023_0306_NCEM"
count = 0

# Authenticate and initialize the Earth Engine API
ee.Authenticate()
ee.Initialize(project=project_name)

# Load GeoJSON
with open('mapbox_grid_1km_filtered_7provinces_ncem(1).geojson') as f:
    geojson = json.load(f)

# Cloud mask for Sentinel-2 SR (using QA60 band)
def maskS2clouds(image):
    qa = image.select('QA60')
    cloudBitMask = 1 << 10
    cirrusBitMask = 1 << 11
    mask = qa.bitwiseAnd(cloudBitMask).eq(0).And(qa.bitwiseAnd(cirrusBitMask).eq(0))
    return image.updateMask(mask).divide(10000)

# Export function for least-cloud image
def export_image(cell_code, row, col, poly_coords, region, from_date, to_date, drive_folder):
    global count

    # Load collection and filter
    collection = ee.ImageCollection('COPERNICUS/S2_HARMONIZED') \
        .filterBounds(region) \
        .filterDate(from_date, to_date) \
        .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20))  # Filter basic cloud %

    # Function to calculate regional cloudiness using QA60 band
    def computeRegionalCloudiness(img):
        cloud_mask = img.select('QA60').bitwiseAnd(1 << 10).Or(
                      img.select('QA60').bitwiseAnd(1 << 11))
        cloud_fraction = cloud_mask.reduceRegion(
            reducer=ee.Reducer.mean(),
            geometry=region,
            scale=10,
            maxPixels=1e13
        ).get('QA60')
        return img.set('regional_cloud', cloud_fraction)

    # Compute regional cloudiness
    with_cloud_scores = collection.map(computeRegionalCloudiness)

    # Sort by regional cloudiness (ascending)
    sorted = with_cloud_scores.sort('regional_cloud')

    first_image = ee.Image(sorted.first())

    # Check if image is valid (has bands)
    band_names = first_image.bandNames().getInfo()
    expected_bands = ['B1','B2','B3','B4','B5','B6','B7','B8','B8A','B9','B10','B11','B12']
    if not all(b in band_names for b in expected_bands):
        print(f"Skipped (no valid bands): {cell_code}_{row}_{col}_0710_2023")
        return

    masked_image = maskS2clouds(first_image).select(expected_bands).clip(region)

    filename = f"{cell_code}_{row}_{col}_0710_2023"
    task = ee.batch.Export.image.toDrive(
        image=masked_image,
        description=filename,
        folder=drive_folder,
        scale=10,
        region=region,
        maxPixels=1e13,
        fileFormat='GeoTIFF',
        formatOptions={'cloudOptimized': True}
    )
    task.start()
    print(f"Exporting: {filename}")
    count += 1

# Date range
from_date = '2023-03-01'
to_date = '2023-07-01'

# Loop through features
for feature in geojson['features']:
    props = feature['properties']
    coords = feature['geometry']['coordinates'][0]

    if coords[0] != coords[-1]:
        coords.append(coords[0])

    poly_coords = [[lon, lat] for lon, lat in coords]
    region = ee.Geometry.Polygon([poly_coords])

    export_image(props['cell_code'], props['row'], props['col'], poly_coords, region, from_date, to_date, drive_folder)

print(count)



*** Earth Engine *** Share your feedback by taking our Annual Developer Satisfaction Survey: https://google.qualtrics.com/jfe/form/SV_7TDKVSyKvBdmMqW?ref=4i2o6


Exporting: 6653_18_312_0710_2023
Exporting: 7022_19_308_0710_2023
Exporting: 7023_19_309_0710_2023
Exporting: 7024_19_310_0710_2023
Exporting: 7025_19_311_0710_2023
Exporting: 7026_19_312_0710_2023
Exporting: 7027_19_313_0710_2023
Exporting: 7395_20_308_0710_2023
Exporting: 7396_20_309_0710_2023
Exporting: 7397_20_310_0710_2023
Exporting: 7398_20_311_0710_2023
Exporting: 7399_20_312_0710_2023
Exporting: 7400_20_313_0710_2023
Exporting: 7769_21_309_0710_2023
Exporting: 7770_21_310_0710_2023
Exporting: 7771_21_311_0710_2023
Exporting: 7772_21_312_0710_2023
Exporting: 7773_21_313_0710_2023
Exporting: 7774_21_314_0710_2023
Exporting: 8144_22_311_0710_2023
Exporting: 8145_22_312_0710_2023
Exporting: 8146_22_313_0710_2023
Exporting: 8147_22_314_0710_2023
Exporting: 8518_23_312_0710_2023
Exporting: 8519_23_313_0710_2023
Exporting: 8520_23_314_0710_2023
Exporting: 8521_23_315_0710_2023
Exporting: 8892_24_313_0710_2023
Exporting: 8893_24_314_0710_2023
Exporting: 8894_24_315_0710_2023
Exporting:

ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.



Exporting: 25843_70_106_0710_2023



KeyboardInterrupt



## Full 7 north provinces sat crawl

In [ ]:
import ee

project_name = "sate-460208"
drive_folder = "SateImg_7NorthProvinces"

# Authenticate and initialize Earth Engine
ee.Authenticate()
ee.Initialize(project=project_name)

# --- Bounding Rectangle Polygon ---
poly_coords = [
    [105.256198491065, 20.522328117615],
    [108.080408864951, 20.522328117615],
    [108.080408864951, 21.6830199181585],
    [105.256198491065, 21.6830199181585],
    [105.256198491065, 20.522328117615]
]
region = ee.Geometry.Polygon([poly_coords])

# Cloud mask for Sentinel-2 SR
def maskS2clouds(image):
    qa = image.select('QA60')
    cloudBitMask = 1 << 10
    cirrusBitMask = 1 << 11
    mask = qa.bitwiseAnd(cloudBitMask).eq(0).And(qa.bitwiseAnd(cirrusBitMask).eq(0))
    return image.updateMask(mask).divide(10000)

# Export function
def export_image(cell_code, region, from_date, to_date, drive_folder):
    collection = ee.ImageCollection('COPERNICUS/S2_HARMONIZED') \
        .filterBounds(region) \
        .filterDate(from_date, to_date) \
        .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 10))

    def computeRegionalCloudiness(img):
        cloud_mask = img.select('QA60').bitwiseAnd(1 << 10).Or(
                      img.select('QA60').bitwiseAnd(1 << 11))
        cloud_fraction = cloud_mask.reduceRegion(
            reducer=ee.Reducer.mean(),
            geometry=region,
            scale=10,
            maxPixels=1e13
        ).get('QA60')
        return img.set('regional_cloud', cloud_fraction)

    sorted = collection.map(computeRegionalCloudiness).sort('regional_cloud')
    first_image = ee.Image(sorted.first())

    expected_bands = ['B1','B2','B3','B4','B5','B6','B7','B8','B8A','B9','B10','B11','B12']
    band_names = first_image.bandNames().getInfo()
    if not all(b in band_names for b in expected_bands):
        print(f"Skipped: {cell_code} (missing bands)")
        return

    masked_image = maskS2clouds(first_image).select(expected_bands).clip(region)

    task = ee.batch.Export.image.toDrive(
        image=masked_image,
        description=cell_code,
        folder=drive_folder,
        scale=10,
        region=region,
        maxPixels=1e13,
        fileFormat='GeoTIFF',
        formatOptions={'cloudOptimized': True}
    )
    task.start()
    print(f"Exporting: {cell_code}")

# Date range
from_date = '2023-09-01'
to_date = '2023-11-01'

# Export the bounding box
export_image("big_rectangle_bbox", region, from_date, to_date, drive_folder)

Exporting: big_rectangle_bbox


## New Air 360

In [ ]:
import json
import ee

project_name = "sate-460208"

# Authenticate and initialize Earth Engine
ee.Authenticate()
ee.Initialize(project=project_name)

# Google Drive export folder
drive_folder = "SateImg3km_0112_2024"

# Load GeoJSON file and extract coordinates
with open("mapbox_grid_3km_filtered_7provinces.json", "r") as f:
    geojson = json.load(f)

# Extract polygons using their coordinates and properties for naming
grid_rectangles = []
for feature in geojson['features']:
    coordinates = feature['geometry']['coordinates']
    props = feature['properties']
    name = f"{props['cell_code']}_{props['row']}_{props['col']}_0112_2024"
    polygon = ee.Geometry.Polygon(coordinates)
    grid_rectangles.append({
        "rectangle": polygon,
        "name": name
    })

# Sentinel-2 cloud masking function
def maskS2clouds(image):
    qa = image.select('QA60')
    cloudBitMask = 1 << 10
    cirrusBitMask = 1 << 11
    mask = qa.bitwiseAnd(cloudBitMask).eq(0).And(qa.bitwiseAnd(cirrusBitMask).eq(0))
    return image.updateMask(mask).divide(10000)

# Prepare export tasks
def prepare_export_tasks(place_name: str, rectangles, from_date: str, to_date: str):
    tasks = []
    for rectangle in rectangles:
        rect = rectangle['rectangle']
        name = rectangle['name']

        image = ee.ImageCollection('COPERNICUS/S2_HARMONIZED') \
            .filterBounds(rect) \
            .filterDate(from_date, to_date) \
            .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 100)) \
            .map(maskS2clouds) \
            .median() \
            .select(['B1','B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A', 'B9', 'B10', 'B11', 'B12']) \
            .clip(rect)

        task = ee.batch.Export.image.toDrive(
            image=image,
            description=f"{place_name}_{name}",
            folder=drive_folder,
            scale=10,
            region=rect,
            maxPixels=1e9,
            fileFormat='GeoTIFF',
            formatOptions={'cloudOptimized': True}
        )
        tasks.append(task)
    return tasks

# Run the tasks
tasks = prepare_export_tasks('7Provinces', grid_rectangles, '2024-01-01', '2024-12-31')
for task in tasks:
    task.start()
    print(f"Exporting {task.config['description']}")


Exporting 7Provinces_355_3_101_0112_2024
Exporting 7Provinces_356_3_102_0112_2024
Exporting 7Provinces_357_3_103_0112_2024
Exporting 7Provinces_483_4_102_0112_2024
Exporting 7Provinces_484_4_103_0112_2024
Exporting 7Provinces_485_4_104_0112_2024
Exporting 7Provinces_612_5_104_0112_2024
Exporting 7Provinces_2321_19_35_0112_2024
Exporting 7Provinces_2322_19_36_0112_2024
Exporting 7Provinces_2446_20_33_0112_2024
Exporting 7Provinces_2447_20_34_0112_2024
Exporting 7Provinces_2448_20_35_0112_2024
Exporting 7Provinces_2449_20_36_0112_2024
Exporting 7Provinces_2571_21_31_0112_2024
Exporting 7Provinces_2572_21_32_0112_2024
Exporting 7Provinces_2573_21_33_0112_2024
Exporting 7Provinces_2574_21_34_0112_2024
Exporting 7Provinces_2575_21_35_0112_2024
Exporting 7Provinces_2576_21_36_0112_2024
Exporting 7Provinces_2584_21_44_0112_2024
Exporting 7Provinces_2585_21_45_0112_2024
Exporting 7Provinces_2586_21_46_0112_2024
Exporting 7Provinces_2587_21_47_0112_2024
Exporting 7Provinces_2599_21_59_0112_2024

## Sentinel-1

In [ ]:
import ee
import os
import ast
import unicodedata
import re

# === KHỞI TẠO EARTH ENGINE ===
ee.Authenticate()
ee.Initialize(project=project_name)

# === CẤU HÌNH ===
count = 0
polygon_folder = '/content/polygon'  # 🔁 Cập nhật thư mục của bạn
from_date = '2024-01-01'
to_date = '2024-12-31'
suffix_to_strip = ''

# === HÀM XỬ LÝ TÊN ===
def strip_accents(text):
    text = text.replace("Đ", "D").replace("đ", "d")
    return ''.join(
        c for c in unicodedata.normalize('NFKD', text)
        if not unicodedata.combining(c)
    )

def sanitize_description(name):
    name = strip_accents(name)
    return re.sub(r'[^a-zA-Z0-9.,:_-]', '_', name)[:100]

# === HÀM EXPORT SENTINEL-1 ===
def export_image(place_name: str, region, from_date: str, to_date: str):
    global count
    description = sanitize_description(f"{place_name}_S1_full_2024")

    image = (ee.ImageCollection('COPERNICUS/S1_GRD')
    .filterBounds(region)
    .filterDate(from_date, to_date)
    .filter(ee.Filter.eq('instrumentMode', 'IW'))
    .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV'))
    .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH'))
    .filter(ee.Filter.eq('orbitProperties_pass', 'DESCENDING'))  # hoặc 'ASCENDING', hoặc bỏ dòng này
    .median()
    .select(['VV', 'VH'])
    .clip(region)
)

    task = ee.batch.Export.image.toDrive(
        image=image,
        description=description,
        folder=drive_folder,
        scale=10,
        region=region,
        maxPixels=1e13,
        fileFormat='GeoTIFF',
        formatOptions={'cloudOptimized': True}
    )

    task.start()
    print(f"🚀 Bắt đầu xuất: {description}")
    count += 1

# === DUYỆT QUA CÁC FILE POLYGON ===
for filename in os.listdir(polygon_folder):
    if filename.endswith('.txt'):
        base_name = os.path.splitext(filename)[0]

        # Tên không dấu cho xuất file
        place_name = base_name.replace(suffix_to_strip, '') if base_name.endswith(suffix_to_strip) else base_name
        clean_place_name = strip_accents(place_name)

        # Đọc polygon
        txt_path = os.path.join(polygon_folder, filename)
        with open(txt_path, 'r', encoding='utf-8') as f:
            coords = ast.literal_eval(f.read())

        # Tạo geometry và export
        region = ee.Geometry.Polygon([coords])
        export_image(clean_place_name, region, from_date, to_date)

print(count)


🚀 Bắt đầu xuất: HaiDuong_S1_full_2024
1


# Get Dataset


In [ ]:
### INITIALIZE VARIABLES ###
project_name = "sate-460208"  # Google Cloud project
drive_folder = "Sentinel-1 New"  # Folder in Google Drive
width = 0.05
height = 0.05

import ee
# Authenticate and initialize the Earth Engine API
ee.Authenticate()
ee.Initialize(project=project_name)

import json
import numpy as np
import matplotlib.pyplot as plt
from shapely.geometry import Polygon, MultiPolygon
from shapely.ops import unary_union
import matplotlib.patches as patches

def parse_multipolygon(multipolygon_wkt):
    """Parse WKT MultiPolygon to Shapely MultiPolygon."""
    import re
    coords_str = re.findall(r'\(([^()]+)\)', multipolygon_wkt)[-1]
    coords = [list(map(float, pair.split())) for pair in coords_str.split(',')]
    return MultiPolygon([Polygon(coords)])

def slice_multipolygon(multipolygon, grid_size=0.05):
    """Slice a MultiPolygon into grid rectangles of specified size."""
    minx, miny, maxx, maxy = multipolygon.bounds
    minx = np.floor(minx / grid_size) * grid_size
    miny = np.floor(miny / grid_size) * grid_size
    maxx = np.ceil(maxx / grid_size) * grid_size
    maxy = np.ceil(maxy / grid_size) * grid_size

    grid_rectangles = []
    x = minx
    while x < maxx:
        y = miny
        while y < maxy:
            rect = Polygon([
                (x, y),
                (x + grid_size, y),
                (x + grid_size, y + grid_size),
                (x, y + grid_size)
            ])
            intersection = rect.intersection(multipolygon)
            if not intersection.is_empty:
                grid_rectangles.append({
                    'rectangle': ee.Geometry.Rectangle([x, y, x + grid_size, y + grid_size]),
                    'position': (x, y)
                })
            y += grid_size
        x += grid_size
    return grid_rectangles

# === Prepare Sentinel-1 Export Tasks ===
def prepare_export_tasks_s1(place_name: str, rectangles, from_date: str, to_date: str):
    tasks = []
    for index, rectangle in enumerate(rectangles):
        rect = rectangle['rectangle']

        image = (ee.ImageCollection('COPERNICUS/S1_GRD')
                 .filterBounds(rect)
                 .filterDate(from_date, to_date)
                 .filter(ee.Filter.eq('instrumentMode', 'IW'))
                 .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV'))
                 .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH'))
                 .filter(ee.Filter.eq('orbitProperties_pass', 'DESCENDING'))
                 .median()
                 .select(['VV', 'VH'])
                 .clip(rect))

        # ✅ File name format: Area_2024_N_o_<index>_sat
        desc = f"{place_name}_2024_N_o_{index}_sat"

        task = ee.batch.Export.image.toDrive(
            image=image,
            description=desc,
            folder=drive_folder,
            scale=10,
            region=rect,
            maxPixels=1e13,
            fileFormat='GeoTIFF',
            formatOptions={'cloudOptimized': True}
        )
        tasks.append(task)
    return tasks


# === Example Run ===
multipolygon_wkt = """
    MULTIPOLYGON (((105.78835682561 21.3687531851594,105.759025263863 21.4190358624393,105.666840355517 21.4776989859324,105.553704331637 21.5908350098121,105.478280315718 21.5615034480655,105.390285630478 21.532171886319,105.293910499025 21.5363621094256,105.281339829705 21.4693185397191,105.348383399411 21.385514077586,105.377714961158 21.4022749700126,105.415426969118 21.3142802847729,105.369334514945 21.3268509540928,105.314861614558 21.2891389461329,105.327432283878 21.2220953764265,105.256198491065 21.1927638146799,105.256198491065 21.0921984601201,105.302290945238 21.008393997987,105.423807415331 20.9874428824538,105.469899869504 20.9832526593471,105.461519423291 20.912018866534,105.457329200184 20.8449752968275,105.511802100571 20.7988828426543,105.545323885424 20.7569806115877,105.574655447171 20.7150783805212,105.612367455131 20.6815565956679,105.645889239984 20.6773663725613,105.666840355517 20.6312739183881,105.679411024837 20.5893716873215,105.733883925224 20.5851814642149,105.759025263863 20.5600401255749,105.792547048717 20.522328117615,105.85121017221 20.6270836952814,105.859590618423 20.6522250339213,105.922443965023 20.6606054801346,105.981107088516 20.6731761494546,106.006248427156 20.6983174880945,106.035579988903 20.656415257028,106.052340881329 20.5893716873215,106.131955120356 20.6061325797481,106.152906235889 20.6396543646014,106.152906235889 20.656415257028,106.190618243849 20.6522250339213,106.219949805596 20.6522250339213,106.270232482875 20.6899370418812,106.286993375302 20.7025077112012,106.374988060542 20.7108881574145,106.387558729862 20.7150783805212,106.421080514715 20.6354641414947,106.467172968888 20.5684205717882,106.550977431021 20.6019423566415,106.638972116261 20.6187032490681,106.731157024607 20.6312739183881,106.764678809461 20.6773663725613,106.831722379167 20.7025077112012,106.802390817421 20.7695512809077,106.798200594314 20.7988828426543,106.835912602274 20.8282144044009,106.87781483334 20.7569806115877,106.944858403047 20.7695512809077,106.982570411007 20.7234588267345,107.032853088287 20.6773663725613,107.125037996633 20.7108881574145,107.125037996633 20.7486001653744,107.21722290498 20.7360294960544,107.242364243619 20.777931727121,107.317788259539 20.8072632888676,107.351310044392 20.7569806115877,107.477016737592 20.668985926348,107.535679861085 20.7737415040143,107.523109191765 20.8156437350809,107.565011422832 20.9078286434273,107.590152761472 20.9413504282806,107.602723430792 20.9874428824538,107.640435438752 21.0712473445869,107.690718116032 21.1341006911867,107.757761685738 21.2011442608932,107.845756370978 21.263997607493,107.921180386898 21.3100900616662,107.958892394858 21.3310411771995,108.042696856991 21.3897043006927,108.080408864951 21.4609380935058,108.017555518351 21.6159763484521,107.988223956604 21.6243567946654,108.000794625924 21.5698838942789,107.942131502431 21.6075959022388,107.916990163791 21.6327372408787,107.883468378938 21.6578785795186,107.837375924765 21.6788296950519,107.761951908845 21.6662590257319,107.707479008458 21.6494981333053,107.673957223605 21.628547017772,107.640435438752 21.6201665715587,107.577582092152 21.6159763484521,107.548250530405 21.6159763484521,107.531489637979 21.6159763484521,107.506348299339 21.641117687092,107.485397183805 21.6578785795186,107.443494952739 21.6830199181585,107.389022052352 21.6243567946654,107.334549151966 21.6034056791321,107.225603351193 21.5950252329188,107.196271789446 21.5740741173855,107.19208156634 21.532171886319,107.200462012553 21.4818892090391,107.208842458766 21.4651283166125,107.21722290498 21.4441772010792,107.225603351193 21.4106554162259,107.200462012553 21.3897043006927,107.171130450806 21.3897043006927,107.133418442846 21.385514077586,107.095706434887 21.385514077586,107.070565096247 21.3771336313727,107.037043311393 21.3603727389461,107.020282418967 21.3436118465195,106.99095085722 21.3226607309862,106.96580951858 21.2975193923463,106.95323884926 21.276568276813,106.944858403047 21.2472367150664,106.944858403047 21.2179051533198,106.9155268413 21.2011442608932,106.898765948874 21.1969540377865,106.848483271594 21.1969540377865,106.764678809461 21.1927638146799,106.764678809461 21.1718126991466,106.764678809461 21.1718126991466,106.710205909074 21.1760029222533,106.638972116261 21.1843833684666,106.571928546555 21.2053344839998,106.538406761701 21.2220953764265,106.530026315488 21.2346660457464,106.500694753741 21.2430464919597,106.488124084421 21.2430464919597,106.442031630248 21.2430464919597,106.383368506755 21.2388562688531,106.349846721902 21.2220953764265,106.337276052582 21.2053344839998,106.328895606369 21.1885735915732,106.316324937049 21.1718126991466,106.295373821515 21.1382909142934,106.270232482875 21.1508615836133,106.266042259769 21.1676224760399,106.215759582489 21.1843833684666,106.198998690062 21.1843833684666,106.178047574529 21.1969540377865,106.173857351422 21.2137149302132,106.161286682102 21.2179051533198,106.123574674142 21.2262855995331,106.090052889289 21.2262855995331,106.048150658223 21.251426938173,106.039770212009 21.2556171612797,106.035579988903 21.2598073843864,105.981107088516 21.276568276813,105.93920485745 21.2681878305997,105.918253741916 21.3058998385596,105.897302626383 21.3519922927328,105.87635151085 21.3687531851594,105.86378084153 21.3813238544794,105.855400395317 21.3938945237993,105.821878610463 21.398084746906,105.779976379397 21.3938945237993,105.77578615629 21.3938945237993,105.78835682561 21.3687531851594)))
    """
# Parse and slice
mpoly = parse_multipolygon(multipolygon_wkt)
grid_rectangles = slice_multipolygon(mpoly)

# Prepare tasks
tasks = prepare_export_tasks_s1('Area', grid_rectangles, '2024-01-01', '2024-12-31')

# Start tasks
for task in tasks:
    task.start()
    print(f"🚀 Exporting {task.config['description']}")


*** Earth Engine *** Share your feedback by taking our Annual Developer Satisfaction Survey: https://google.qualtrics.com/jfe/form/SV_7TDKVSyKvBdmMqW?ref=4i2o6


🚀 Exporting Area_2024_N_o_0_sat
🚀 Exporting Area_2024_N_o_1_sat
🚀 Exporting Area_2024_N_o_2_sat
🚀 Exporting Area_2024_N_o_3_sat
🚀 Exporting Area_2024_N_o_4_sat
🚀 Exporting Area_2024_N_o_5_sat
🚀 Exporting Area_2024_N_o_6_sat
🚀 Exporting Area_2024_N_o_7_sat
🚀 Exporting Area_2024_N_o_8_sat
🚀 Exporting Area_2024_N_o_9_sat
🚀 Exporting Area_2024_N_o_10_sat
🚀 Exporting Area_2024_N_o_11_sat
🚀 Exporting Area_2024_N_o_12_sat
🚀 Exporting Area_2024_N_o_13_sat
🚀 Exporting Area_2024_N_o_14_sat
🚀 Exporting Area_2024_N_o_15_sat
🚀 Exporting Area_2024_N_o_16_sat
🚀 Exporting Area_2024_N_o_17_sat
🚀 Exporting Area_2024_N_o_18_sat
🚀 Exporting Area_2024_N_o_19_sat
🚀 Exporting Area_2024_N_o_20_sat
🚀 Exporting Area_2024_N_o_21_sat
🚀 Exporting Area_2024_N_o_22_sat
🚀 Exporting Area_2024_N_o_23_sat
🚀 Exporting Area_2024_N_o_24_sat
🚀 Exporting Area_2024_N_o_25_sat
🚀 Exporting Area_2024_N_o_26_sat
🚀 Exporting Area_2024_N_o_27_sat
🚀 Exporting Area_2024_N_o_28_sat
🚀 Exporting Area_2024_N_o_29_sat
🚀 Exporting Area_202